# SOAC Library Usage Example

This notebook demonstrates how to use the `soac` library to find integer solutions to Diophantine equations.

In [2]:
import numpy as np
import time
from soac import run_soac
from soac.problems import F_1, F_2a, F_3a, F_4, F_7, F_8

## Example 1: Simple Linear Equation (Problem 1)

Find integer solutions to `15x + 11y = 12`.

In [3]:
n = 2
bounds = (np.array([-50.0, -50.0]), np.array([50.0, 50.0]))

params = {
    'm_cluster': 350,
    'k_cluster': 10,
    'gamma': 0.01,
    'epsilon': 1e-7,
    'delta': 0.1,
    'm': 30,
    'k_max': 10,
    'r': 0.95,
    'theta': np.pi / 4
}

start = time.time()
roots = run_soac(n, bounds, params, F_1)
elapsed = time.time() - start

print(f"Found {len(roots)} roots in {elapsed:.4f} seconds:")
for r in roots:
    print(f"  x={int(r[0])}, y={int(r[1])}  ->  15*{int(r[0])} + 11*{int(r[1])} = {15*int(r[0]) + 11*int(r[1])}")

Found 7 roots in 0.1507 seconds:
  x=14, y=-18  ->  15*14 + 11*-18 = 12
  x=3, y=-3  ->  15*3 + 11*-3 = 12
  x=36, y=-48  ->  15*36 + 11*-48 = 12
  x=-30, y=42  ->  15*-30 + 11*42 = 12
  x=-8, y=12  ->  15*-8 + 11*12 = 12
  x=-19, y=27  ->  15*-19 + 11*27 = 12
  x=25, y=-33  ->  15*25 + 11*-33 = 12


## Example 2: Sum of Squares (Problem 2a)

Find integers `x_1, ..., x_9` such that `sum(x_i^2) = 720`.

In [4]:
n = 9
bounds = (np.ones(n), 26 * np.ones(n))

params = {
    'm_cluster': 1000,
    'k_cluster': 20,
    'gamma': 0.1,
    'epsilon': 1e-5,
    'delta': 0.01,
    'm': 50,
    'k_max': 50,
    'r': 0.95,
    'theta': np.pi / 4
}

start = time.time()
roots = run_soac(n, bounds, params, F_2a)
elapsed = time.time() - start

print(f"Found {len(roots)} roots in {elapsed:.4f} seconds:")
for i, r in enumerate(roots[:5]):
    r_int = [int(v) for v in r]
    print(f"  Root {i+1}: {r_int}  ->  sum of squares = {sum(v**2 for v in r_int)}")
if len(roots) > 5:
    print(f"  ... and {len(roots) - 5} more")

Found 153 roots in 4.6208 seconds:
  Root 1: [4, 15, 1, 1, 11, 3, 11, 1, 15]  ->  sum of squares = 720
  Root 2: [3, 2, 3, 6, 16, 2, 13, 8, 13]  ->  sum of squares = 720
  Root 3: [3, 7, 14, 6, 11, 14, 3, 2, 10]  ->  sum of squares = 720
  Root 4: [2, 14, 3, 5, 11, 4, 4, 3, 18]  ->  sum of squares = 720
  Root 5: [3, 1, 10, 8, 14, 13, 12, 1, 6]  ->  sum of squares = 720
  ... and 148 more


## Example 3: Cubic Sum (Problem 3a)

Find integers `x, y` such that `x^3 + y^3 = 1008`.

In [5]:
n = 2
bounds = (np.ones(n), 10 * np.ones(n))

params = {
    'm_cluster': 200,
    'k_cluster': 20,
    'gamma': 0.1,
    'epsilon': 1e-5,
    'delta': 0.01,
    'm': 50,
    'k_max': 50,
    'r': 0.95,
    'theta': np.pi / 4
}

start = time.time()
roots = run_soac(n, bounds, params, F_3a)
elapsed = time.time() - start

print(f"Found {len(roots)} roots in {elapsed:.4f} seconds:")
for r in roots:
    x, y = int(r[0]), int(r[1])
    print(f"  ({x}, {y})  ->  {x}^3 + {y}^3 = {x**3 + y**3}")

Found 2 roots in 0.0601 seconds:
  (2, 10)  ->  2^3 + 10^3 = 1008
  (10, 2)  ->  10^3 + 2^3 = 1008


## Writing Your Own Fitness Function

The `soac` library accepts any fitness function `F: R^n -> [0, 1]` where `F(x) = 1` iff `x` is an exact integer solution.

The standard pattern is:
```python
def my_fitness(x):
    residual = <compute equation residual using x>
    return 1.0 / (1.0 + abs(residual))
```

For systems of equations, sum the absolute residuals:
```python
def my_system_fitness(x):
    f1 = <equation 1 residual>
    f2 = <equation 2 residual>
    return 1.0 / (1.0 + abs(f1) + abs(f2))
```

In [6]:
# Example: Find integers (x, y) such that x^2 + y^2 = 50
def my_fitness(x):
    val = x[0]**2 + x[1]**2 - 50
    return 1.0 / (1.0 + abs(val))

n = 2
bounds = (np.array([-20.0, -20.0]), np.array([20.0, 20.0]))

params = {
    'm_cluster': 200,
    'k_cluster': 15,
    'gamma': 0.01,
    'epsilon': 1e-7,
    'delta': 0.1,
    'm': 30,
    'k_max': 20,
    'r': 0.95,
    'theta': np.pi / 4
}

start = time.time()
roots = run_soac(n, bounds, params, my_fitness)
elapsed = time.time() - start

print(f"Found {len(roots)} roots in {elapsed:.4f} seconds:")
for r in roots:
    x, y = int(r[0]), int(r[1])
    print(f"  ({x}, {y})  ->  {x}^2 + {y}^2 = {x**2 + y**2}")

Found 11 roots in 0.1431 seconds:
  (1, 7)  ->  1^2 + 7^2 = 50
  (7, -1)  ->  7^2 + -1^2 = 50
  (5, -5)  ->  5^2 + -5^2 = 50
  (-7, -1)  ->  -7^2 + -1^2 = 50
  (1, -7)  ->  1^2 + -7^2 = 50
  (-1, -7)  ->  -1^2 + -7^2 = 50
  (5, 5)  ->  5^2 + 5^2 = 50
  (-1, 7)  ->  -1^2 + 7^2 = 50
  (-7, 1)  ->  -7^2 + 1^2 = 50
  (-5, 5)  ->  -5^2 + 5^2 = 50
  (-5, -5)  ->  -5^2 + -5^2 = 50


## Parameter Tuning Tips

Key parameters and their effects:

| Parameter | Description | Typical Range |
|-----------|-------------|---------------|
| `m_cluster` | More points = better coverage but slower | 100 - 30000 |
| `k_cluster` | More iterations = better clustering | 10 - 50 |
| `gamma` | Fitness threshold for clustering | 0.001 - 0.1 |
| `epsilon` | Solution acceptance tolerance | 1e-7 - 0.001 |
| `delta` | Duplicate merging distance | 0.001 - 0.1 |
| `m` | Points per cluster in spiral phase | 20 - 200 |
| `k_max` | Spiral iterations per cluster | 10 - 50 |
| `r` | Contraction rate (must be < 1) | 0.9 - 0.99 |
| `theta` | Rotation angle | pi/60 - pi/4 |

For quick exploration, start with smaller `m_cluster` and `m`. Increase for harder problems.